# 01 - Coleta de dados

Notebook responsável por buscar os dados brutos (clima e produção cafeeira
da região de Lavras-MG) nas fontes originais (APIs, downloads, etc.) e
salvá-los sem alterações em `data/raw/`.

Usa as funções de `src/coleta.py`.

## Clima (Open-Meteo)

Temperatura máxima/mínima e precipitação diárias para Lavras-MG, 1974-2024. Ampliamos de 2000-2024 para 1974-2024 depois de descobrir que a série do IBGE/SIDRA cobre esse período inteiro — mais anos significa mais chance de capturar geadas históricas reais (ex. a geada de 1975 no Sul de Minas) e uma amostra maior para testar a correlação clima x produção (ver `03_eda.ipynb`).

In [1]:
import sys
sys.path.append("..")

from src.coleta import coletar_clima, coletar_producao_cafe, coletar_geada_estacao_83687, validar_dados

df_clima = coletar_clima(1974, 2024)
df_clima.shape


200
Arquivo salvo em data/raw/lavras_clima_1974_2024.csv


(18628, 4)

In [2]:
validar_dados(df_clima, coluna_data="time")

Número de linhas: 18628
Primeira data: 1974-01-01
Última data: 2024-12-31
Valores faltando por coluna:
time                  0
temperature_2m_max    0
temperature_2m_min    0
precipitation_sum     0
dtype: int64


## Produção cafeeira (IBGE - Produção Agrícola Municipal)

Série anual de área colhida, quantidade produzida e rendimento médio de café em Lavras-MG, 1974-2024. Fonte: tabela 1613 do SIDRA/IBGE (PAM), produto "Café (em grão) Total", nível municipal, acesso público via API sem necessidade de login.

Avaliamos antes a CONAB, mas suas séries históricas de café são por região produtora (ex. "Sul de Minas"), não por município — não isolam Lavras especificamente. O IBGE/PAM tem granularidade municipal, o que combina com os dados climáticos que já são pontuais para Lavras.

In [3]:
df_cafe = coletar_producao_cafe(1974, 2024)
df_cafe


200
Arquivo salvo em data/raw/lavras_producao_cafe_1974_2024.csv


,ano,quantidade_produzida_t,rendimento_medio_kg_ha,area_colhida_ha
0,1974,2880,2000,1440
1,1975,1197,750,1596
2,1976,3302,1914,1725
3,1977,3120,1600,1950
4,1978,4208,1600,2630
5,1979,4715,1600,2947
6,1980,383,411,930
7,1981,8270,3757,2201
8,1982,1507,807,1868
9,1983,3140,1434,2190


In [4]:
validar_dados(df_cafe, coluna_data="ano")

Número de linhas: 51
Primeira data: 1974
Última data: 2024
Valores faltando por coluna:
ano                       0
quantidade_produzida_t    0
rendimento_medio_kg_ha    0
area_colhida_ha           0
dtype: int64


## Geada: estação real 83687 (Lavras/UFLA, via BR-DWGD)

Temperatura mínima diária **observada** na estação convencional do INMET em Lavras/UFLA (código 83687, desde 1911) — não é reanálise em grade como o Open-Meteo acima. Extraída dos dados de observação de estação usados para construir o BR-DWGD (Xavier et al., 2022), disponibilizados publicamente sem necessidade de conta/login: https://github.com/AlexandreCandidoXavier/BR-DWGD.

Motivação: o Open-Meteo (reanálise, ~25-31km de grade) nunca registrou geada na série original de 2000-2024 — ver investigação em `03_eda.ipynb`. Essa é a fonte de dado de estação real que buscávamos desde então (tentamos BDMEP e a API/portal do INMET sem sucesso; esta veio pela mesma base de dados usada pelo BR-DWGD). Com só 25 anos, a estação registrou apenas 1 dia de geada — usamos 1974-2024 aqui para tentar capturar mais eventos.

In [5]:
df_geada = coletar_geada_estacao_83687(1974, 2024)
df_geada.shape


Arquivo salvo em data/raw/lavras_geada_estacao_83687_1974_2024.csv


(18628, 2)

In [6]:
validar_dados(df_geada, coluna_data="time")


Número de linhas: 18628
Primeira data: 1974-01-01 00:00:00
Última data: 2024-12-31 00:00:00
Valores faltando por coluna:
time              0
tmin_estacao    626
dtype: int64
